# Olist E-Commerce Data Understanding

## Objective                          

In [1]:
import pandas as pd
from pathlib import Path

print("pandas version: ", pd.__version__)

pandas version:  3.0.6


In [2]:
DATA_DIR = Path("../data/raw")

csv_files = list(DATA_DIR.glob("*.csv"))

for file in csv_files:
    print(file.name)
    

olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


In [3]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")

## Orders Table 

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
orders.shape

(99441, 8)

In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [8]:
len(orders)

99441

In [11]:
orders["order_id"].nunique()

99441

In [13]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [14]:
orders.isna().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [15]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [16]:
orders["order_status"].value_counts(normalize=True) * 100

order_status
delivered      97.020344
shipped         1.113223
canceled        0.628513
unavailable     0.612423
invoiced        0.315765
processing      0.302692
created         0.005028
approved        0.002011
Name: proportion, dtype: float64

In [17]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [18]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
]

delivered_orders.isna().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [19]:
delivered_orders.shape

(96478, 8)

### Initial Observations

- The orders table contains 99,441 orders.
- `order_id` is unique across the table with no duplicate values, making it a candidate primary key.
- Approximately 97.02% of orders have a `delivered` status.
- Missing values are concentrated in the approval, carrier delivery, and customer delivery timestamp fields.
- Some missing delivery timestamps may be expected for orders that were canceled, unavailable, or not yet delivered.
- Further investigation is needed to determine whether delivered orders also contain missing fulfillment timestamps.

## Investigating Missing Order Timestamps

Missing delivery-related timestamps may be expected for orders that were canceled, unavailable, or still in progress. To distinguish expected missingness from potential data-quality issues, missing timestamps are examined by order status.

In [20]:
missing_by_status = (
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"

        ]
    ]
    .apply(lambda x: x.isna().sum())
)

missing_by_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


### Investigatimg only delivered orders

In [25]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()


In [26]:
delivered_orders.shape

(96478, 8)

In [27]:
delivered_orders.isna().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

# Customers Table

In [28]:
customers = pd.read_csv(
    DATA_DIR / "olist_customers_dataset.csv"
)

In [29]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [30]:
customers.shape

(99441, 5)

In [31]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [32]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

### investigating the two customer IDs

In [36]:
customers["customer_id"].nunique()

99441

In [37]:
customers["customer_unique_id"].nunique()

96096

In [38]:
customers["customer_id"].duplicated().sum()

np.int64(0)

### the number of customer_id is greater than the number of customer_unique_id,
- this could be a result of repeat customers with multiple orders


### find repeated customers

In [40]:
#the customers that appear most frequently

customers["customer_unique_id"].value_counts().head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
de34b16117594161a6a89c50b289d35a     6
Name: count, dtype: int64

In [41]:
#customer purchasing behavior data

customers["customer_unique_id"].value_counts().describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: count, dtype: float64

### Understanding the relationship with orders

In [46]:
orders["customer_id"].nunique()

99441

In [45]:
customers["customer_id"].nunique()

99441

### check whether every order's customer_id exists in the customer table

In [47]:
orders["customer_id"].isin(
    customers["customer_id"]
).all()

np.True_

### Customer Geography

In [48]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [49]:
customers["customer_city"].value_counts().head(10)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64

### Customer Table Observations

- The `customer_id` field uniquely identifies records in the customer table.
- `customer_unique_id` represents the underlying customer and can appear more than once when a customer places multiple orders.
- `customer_id` connects the customers table to the orders table.
- Customer geographic information includes city, state, and ZIP-code prefix.
- Geographic information may later be useful for analyzing delivery performance and estimating seller-to-customer shipping distance.
- Customer records are geographically concentrated in São Paulo (SP), which has 41,746 records.
- São Paulo is also the most common customer city, with 15,540 records.
- Because order volume varies substantially across regions, future geographic comparisons should consider delivery rates and sample sizes rather than raw delay counts alone.

## Order Items Table

The order items table contains information about the individual products included in each order. This table will be examined to determine its grain, key structure, and relationship with the orders table.

In [50]:
items = pd.read_csv(
    DATA_DIR / "olist_order_items_dataset.csv"
)

items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [51]:
items.shape

(112650, 7)

In [52]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [53]:
items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [54]:
items["order_id"].duplicated().sum()

np.int64(13984)

### determining table's actual key

In [55]:
items["order_id"].nunique()

98666

In [56]:
items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [57]:
items["order_id"].isin(
    orders["order_id"]
).all()

np.True_

### Order Items Observations

- The table contains 112,650 order-item records representing 98,666 unique orders.
- There are no missing values in the table.
- `order_id` is not unique because a single order can contain multiple items.
- The combination of `order_id` and `order_item_id` uniquely identifies each row and can serve as a composite primary key.
- Every `order_id` in the order items table exists in the orders table, supporting `order_id` as a foreign key.
- The relationship between `orders` and `order_items` is one-to-many.

### Items Per Order

In [58]:
items_per_order = (
    items.groupby("order_id")
         .size()
)

items_per_order.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64